# Generate fact_events for Product Funnel & Retention Analysis

This notebook creates the event-level table (`fact_events`) for a simulated freemium wellness app.

It generates the following events:
- app_install
- sign_up
- onboarding_completed
- session_started
- session_completed
- subscription_started

In [1]:
import pandas as pd
import numpy as np
import uuid
import random
from datetime import timedelta

In [2]:
random.seed(42)
np.random.seed(42)

In [3]:
users_df = pd.read_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/dim_users.csv")
users_df.shape

(10000, 5)

In [4]:
users_df.head()

,user_id,signup_ts,platform,persona,country
0,27df2322-322b-44d5-a6f9-00e75037da2a,2025-01-06 09:42:36,iOS,casual,Germany
1,242b3c5d-f9be-4fac-82a3-1103a9009a83,2025-01-02 05:08:22,Android,one-time,Germany
2,8254d52c-8798-4319-91c2-45eba48fbdbd,2025-01-14 08:25:56,Web,one-time,Nepal
3,6713e50b-97e0-4fc1-a901-ae9d591b2d9d,2025-01-12 21:19:10,Android,casual,Australia
4,9f8c5ec5-3cbb-46b7-888c-7503175d1316,2025-01-11 20:03:33,Web,power,USA


## Step 1: Generate base events

Every user gets:
- app_install
- sign_up

In [5]:
users_input = users_df.copy()
users_input["signup_ts"] = pd.to_datetime(users_input["signup_ts"])

In [6]:
base_events = []

for _, row in users_input.iterrows():
    user_id = row["user_id"]
    signup_ts = row["signup_ts"]
    persona = row["persona"]
    
    # app_install
    base_events.append({
        "event_id": str(uuid.uuid4()),
        "user_id": user_id,
        "event_name": "app_install",
        "signup_ts": signup_ts,
        "persona": persona,
        "session_id": None
    })
    
    # sign_up
    base_events.append({
        "event_id": str(uuid.uuid4()),
        "user_id": user_id,
        "event_name": "sign_up",
        "signup_ts": signup_ts,
        "persona": persona,
        "session_id": None
    })

events_df = pd.DataFrame(base_events)

events_df.head()

,event_id,user_id,event_name,signup_ts,persona,session_id
0,33279142-dd1e-459e-80a3-2e938104a254,27df2322-322b-44d5-a6f9-00e75037da2a,app_install,2025-01-06 09:42:36,casual,None
1,f2f3506f-d5aa-49b2-aa6f-132b4178faee,27df2322-322b-44d5-a6f9-00e75037da2a,sign_up,2025-01-06 09:42:36,casual,None
2,a5761888-0b9b-42dc-8030-b16101e9663e,242b3c5d-f9be-4fac-82a3-1103a9009a83,app_install,2025-01-02 05:08:22,one-time,None
3,9cf392ca-0b0f-41e3-9e71-193e638e4db2,242b3c5d-f9be-4fac-82a3-1103a9009a83,sign_up,2025-01-02 05:08:22,one-time,None
4,4cb73111-b780-4eb4-910a-02eb15aa3919,8254d52c-8798-4319-91c2-45eba48fbdbd,app_install,2025-01-14 08:25:56,one-time,None


## Step 2: Add timestamps

- sign_up occurs at signup_ts
- app_install occurs 30 seconds to 10 minutes before signup_ts

In [7]:
mask_signup = events_df["event_name"].eq("sign_up")

# sign_up happens at signup_ts
events_df["event_ts"] = events_df["signup_ts"]

# app_install happens 30 seconds to 10 minutes before signup
n_installs = (~mask_signup).sum()
offset_seconds = np.random.randint(30, 10 * 60 + 1, size=n_installs)
offset_tds = pd.to_timedelta(offset_seconds, unit="s")

events_df.loc[~mask_signup, "event_ts"] = (
    events_df.loc[~mask_signup, "signup_ts"] - offset_tds
)

events_df["event_date"] = pd.to_datetime(events_df["event_ts"]).dt.date

events_df = events_df.sort_values(["user_id", "event_ts"]).reset_index(drop=True)

events_df.head()

,event_id,user_id,event_name,signup_ts,persona,session_id,event_ts,event_date
0,afa26443-c71a-4e27-8ee9-6235ea790ab6,0000142a-fb28-440d-8be1-30c131aad266,app_install,2025-01-30 22:38:47,one-time,None,2025-01-30 22:33:25,2025-01-30
1,e7bd5af9-6d6b-45ce-a162-bca78d30bfec,0000142a-fb28-440d-8be1-30c131aad266,sign_up,2025-01-30 22:38:47,one-time,None,2025-01-30 22:38:47,2025-01-30
2,30853cff-8fbf-48ba-8dc5-1d8f8917baa0,0007eaaa-bb53-460e-9441-4cf87e41f8db,app_install,2025-01-13 12:39:11,power,None,2025-01-13 12:31:56,2025-01-13
3,f9ea3843-c522-4cee-9389-c1c00f0c866e,0007eaaa-bb53-460e-9441-4cf87e41f8db,sign_up,2025-01-13 12:39:11,power,None,2025-01-13 12:39:11,2025-01-13
4,5edc601b-e913-4344-9bbe-66a56a77d485,00090121-07ef-46c8-8b9c-0875bca124d5,app_install,2025-01-29 02:30:27,casual,None,2025-01-29 02:22:29,2025-01-29


## Step 3: Add onboarding_completed

Onboarding completion probability by persona:
- power: 0.80
- casual: 0.55
- one_time: 0.25

In [8]:
onboarding_prob = {
    "power": 0.80,
    "casual": 0.55,
    "one-time": 0.25
}

users_for_onboarding = users_input[["user_id", "signup_ts", "persona"]].copy()
users_for_onboarding["p_onboard"] = users_for_onboarding["persona"].map(onboarding_prob)

rand = np.random.random(len(users_for_onboarding))
users_for_onboarding["did_onboard"] = rand < users_for_onboarding["p_onboard"]

onboard_users = users_for_onboarding[users_for_onboarding["did_onboard"]].copy()

mins = np.random.randint(1, 16, size=len(onboard_users))
onboard_users["event_ts"] = onboard_users["signup_ts"] + pd.to_timedelta(mins, unit="m")

onboarding_events = pd.DataFrame({
    "event_id": [str(uuid.uuid4()) for _ in range(len(onboard_users))],
    "user_id": onboard_users["user_id"].values,
    "event_name": "onboarding_completed",
    "signup_ts": onboard_users["signup_ts"].values,
    "persona": onboard_users["persona"].values,
    "session_id": None,
    "event_ts": onboard_users["event_ts"].values,
    "event_date": onboard_users["event_ts"].dt.date.values
})

events_df = pd.concat([events_df, onboarding_events], ignore_index=True)
events_df = events_df.sort_values(["user_id", "event_ts"]).reset_index(drop=True)

events_df["event_name"].value_counts()

app_install             10000
sign_up                 10000
onboarding_completed     5125
Name: event_name, dtype: int64

## Step 4: Add session events

Only users who completed onboarding can generate sessions.

Session ranges by persona:
- power: 4–8
- casual: 2–4
- one_time: 0–1

In [9]:
session_events = []

for _, row in users_for_onboarding.iterrows():
    
    if not row["did_onboard"]:
        continue
    
    user_id = row["user_id"]
    signup_ts = row["signup_ts"]
    persona = row["persona"]
    
    if persona == "power":
        n_sessions = np.random.randint(4, 9)   # 4 to 8
    elif persona == "casual":
        n_sessions = np.random.randint(2, 5)   # 2 to 4
    else:
        n_sessions = np.random.randint(0, 2)   # 0 to 1
    
    for _ in range(n_sessions):
        session_id = str(uuid.uuid4())
        
        day_offset = np.random.randint(0, 31)
        second_offset = np.random.randint(0, 24 * 60 * 60)
        
        session_start = signup_ts + timedelta(days=day_offset, seconds=second_offset)
        session_complete = session_start + timedelta(minutes=np.random.randint(3, 26))
        
        session_events.append({
            "event_id": str(uuid.uuid4()),
            "user_id": user_id,
            "event_name": "session_started",
            "signup_ts": signup_ts,
            "persona": persona,
            "session_id": session_id,
            "event_ts": session_start,
            "event_date": session_start.date()
        })
        
        session_events.append({
            "event_id": str(uuid.uuid4()),
            "user_id": user_id,
            "event_name": "session_completed",
            "signup_ts": signup_ts,
            "persona": persona,
            "session_id": session_id,
            "event_ts": session_complete,
            "event_date": session_complete.date()
        })

session_events_df = pd.DataFrame(session_events)

events_df = pd.concat([events_df, session_events_df], ignore_index=True)
events_df = events_df.sort_values(["user_id", "event_ts"]).reset_index(drop=True)

events_df["event_name"].value_counts()

session_started         18445
session_completed       18445
app_install             10000
sign_up                 10000
onboarding_completed     5125
Name: event_name, dtype: int64

## Step 5: Add subscription_started

Only users with at least one completed session are eligible.

Subscription probability by persona:
- power: 0.35
- casual: 0.12
- one_time: 0.02

In [10]:
eligible_users = (
    events_df[events_df["event_name"] == "session_completed"]["user_id"]
    .unique()
)

first_session_df = (
    events_df[events_df["event_name"] == "session_completed"]
    .groupby("user_id", as_index=False)["event_ts"]
    .min()
    .rename(columns={"event_ts": "first_session_ts"})
)

user_lookup = (
    events_df[["user_id", "persona", "signup_ts"]]
    .drop_duplicates("user_id")
    .copy()
)

subs_base = first_session_df.merge(user_lookup, on="user_id", how="left")

sub_prob = {
    "power": 0.35,
    "casual": 0.12,
    "one-time": 0.02
}

subs_base["p_sub"] = subs_base["persona"].map(sub_prob).fillna(0.0)
subs_base["did_subscribe"] = np.random.random(len(subs_base)) < subs_base["p_sub"]

subs_users = subs_base[subs_base["did_subscribe"]].copy()

subscription_events = pd.DataFrame(columns=events_df.columns)

if len(subs_users) > 0:
    offset_minutes = np.random.randint(5, 7 * 24 * 60 + 1, size=len(subs_users))
    subs_users["sub_ts_raw"] = subs_users["first_session_ts"] + pd.to_timedelta(offset_minutes, unit="m")
    subs_users["sub_ts_max"] = subs_users["signup_ts"] + pd.to_timedelta(14, unit="D")
    
    subs_users["sub_ts"] = np.minimum(
        subs_users["sub_ts_raw"].values,
        subs_users["sub_ts_max"].values
    )
    subs_users["sub_ts"] = pd.to_datetime(subs_users["sub_ts"])
    
    subscription_events = pd.DataFrame({
        "event_id": [str(uuid.uuid4()) for _ in range(len(subs_users))],
        "user_id": subs_users["user_id"].values,
        "event_name": "subscription_started",
        "signup_ts": subs_users["signup_ts"].values,
        "persona": subs_users["persona"].values,
        "session_id": None,
        "event_ts": subs_users["sub_ts"].values,
        "event_date": subs_users["sub_ts"].dt.date.values
    })
    
    events_df = pd.concat([events_df, subscription_events], ignore_index=True)
    events_df = events_df.sort_values(["user_id", "event_ts"]).reset_index(drop=True)

events_df["event_name"].value_counts()

session_started         18445
session_completed       18445
app_install             10000
sign_up                 10000
onboarding_completed     5125
subscription_started      909
Name: event_name, dtype: int64

In [11]:
events_df.shape
events_df["event_name"].value_counts()
events_df.head()

,event_id,user_id,event_name,signup_ts,persona,session_id,event_ts,event_date
0,afa26443-c71a-4e27-8ee9-6235ea790ab6,0000142a-fb28-440d-8be1-30c131aad266,app_install,2025-01-30 22:38:47,one-time,None,2025-01-30 22:33:25,2025-01-30
1,e7bd5af9-6d6b-45ce-a162-bca78d30bfec,0000142a-fb28-440d-8be1-30c131aad266,sign_up,2025-01-30 22:38:47,one-time,None,2025-01-30 22:38:47,2025-01-30
2,61010338-a5ac-4c40-acda-faaca62caa49,0000142a-fb28-440d-8be1-30c131aad266,onboarding_completed,2025-01-30 22:38:47,one-time,None,2025-01-30 22:49:47,2025-01-30
3,14daccbc-a999-4101-8ae0-dd975ee12a7e,0000142a-fb28-440d-8be1-30c131aad266,session_started,2025-01-30 22:38:47,one-time,804155c4-ddb6-4cb9-948c-7778d895b083,2025-02-25 02:41:26,2025-02-25
4,bc54b8d1-9ae8-4e54-b43d-a7217bb19632,0000142a-fb28-440d-8be1-30c131aad266,session_completed,2025-01-30 22:38:47,one-time,804155c4-ddb6-4cb9-948c-7778d895b083,2025-02-25 02:46:26,2025-02-25


In [12]:
fact_events = events_df[
    ["event_id", "user_id", "event_name", "event_ts", "event_date", "session_id"]
].copy()

fact_events.shape
fact_events.head()

,event_id,user_id,event_name,event_ts,event_date,session_id
0,afa26443-c71a-4e27-8ee9-6235ea790ab6,0000142a-fb28-440d-8be1-30c131aad266,app_install,2025-01-30 22:33:25,2025-01-30,None
1,e7bd5af9-6d6b-45ce-a162-bca78d30bfec,0000142a-fb28-440d-8be1-30c131aad266,sign_up,2025-01-30 22:38:47,2025-01-30,None
2,61010338-a5ac-4c40-acda-faaca62caa49,0000142a-fb28-440d-8be1-30c131aad266,onboarding_completed,2025-01-30 22:49:47,2025-01-30,None
3,14daccbc-a999-4101-8ae0-dd975ee12a7e,0000142a-fb28-440d-8be1-30c131aad266,session_started,2025-02-25 02:41:26,2025-02-25,804155c4-ddb6-4cb9-948c-7778d895b083
4,bc54b8d1-9ae8-4e54-b43d-a7217bb19632,0000142a-fb28-440d-8be1-30c131aad266,session_completed,2025-02-25 02:46:26,2025-02-25,804155c4-ddb6-4cb9-948c-7778d895b083


In [13]:
fact_events.to_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/fact_events.csv", index=False)

if len(fact_events) >= 500:
    fact_events.sample(500, random_state=42).to_csv(
        "/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/fact_events_sample.csv",
        index=False
    )
else:
    fact_events.to_csv(
        "/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/fact_events_sample.csv",
        index=False
    )

In [14]:
fact_events.shape
fact_events["event_name"].value_counts()

session_started         18445
session_completed       18445
app_install             10000
sign_up                 10000
onboarding_completed     5125
subscription_started      909
Name: event_name, dtype: int64